<a href="https://www.kaggle.com/code/mahsazamanifard/cleaning-and-encoding-text-techniques-nlp?scriptVersionId=96946440" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

**I have recently started learning NLP, This notebook is a collection of blog posts on [machinelearningmastery](https://machinelearningmastery.com/) (with minor changes) about text cleaning and preprocessing; I wanted to create this notebook, as a reference for my future use. hope it can help someone else too! :)**

***PS: The corresponding blog post is mentioned at the beginning of each section***

# Import and Read the Data:

In [ ]:
import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
txt='../input/metamorphed/metamorphosis_clean.txt'
with open(txt,'rt',encoding='utf-8-sig') as f:
    file=f.read()


In [ ]:
file[:100]

# Basic Text Cleaning:

[Blog Post](https://machinelearningmastery.com/clean-text-machine-learning-python/)

## White-Space split:

In [ ]:
words=file.split()

*We can see that punctuation is preserved (e.g. wasn’t and armour-like), which is
nice. We can also see that end of sentence punctuation is kept with the last word (e.g. thought.),which is not great.*

In [ ]:
set(words[:100])

## Regex:

In [ ]:
import re

In [ ]:
# re.split: Returns a list where the string has been split at each match

# \W: Returns a match where the string DOES NOT contain any word characters 
# (characters OTHER THAN from a to Z, digits from 0-9, and the underscore _ character)

# r before'\W+': means the string will be treated as raw string.
# When an 'r' or 'R' prefix is present, a character following a backslash
# is included in the string without change, and all backslashes are left inside
# the string

words_re=re.split(r'\W+',file)

*running the example we can see that we get our list of words. This time, we can see
that armour-like is now two words armour and like but contractions like "What’s" is also
two words What and s.*

In [ ]:
set(words_re[:100])

## Split by Whitespace and Remove Punctuation:

In [ ]:
import string
string.punctuation

*We can use regular expressions to select for the punctuation characters and use the sub()
function to replace them with nothing.*

In [ ]:
words_punc=file.split()

In [ ]:
# prepare regex for char filtering
re_punc = re.compile('[{}]'.format(string.punctuation))

#The re.compile() method
#We can combine a regular expression pattern into pattern objects,
#which can be used for pattern matching. It also helps to search a
#pattern again without rewriting it.

In [ ]:
# remove punctuation from each word
words_stripped = [re_punc.sub('', w) for w in words_punc]

In [ ]:
set(words_stripped[:100])

*We can see that this has had the desired effect, mostly. Contractions like What’s have
become Whats but armour-like has become armourlike.*


## Non-Printable Characters:

*Sometimes text data may contain non-printable characters. We can use a similar approach to
filter out all non-printable characters by selecting the inverse of the string.printable constant.*

In [ ]:
re_print=re.compile('[^{}]'.format(string.printable)) #creating a re object

In [ ]:
# remove punctuation from each word
result = [re_print.sub('', w) for w in words_stripped]

In [ ]:
result[:100]

## Normalizing Case:

*It is common to convert all words to one case. This means that the vocabulary will shrink in
size, but some distinctions are lost (e.g. Apple the company vs apple the fruit is a commonly
used example). We can convert all words to lowercase by calling the lower() function on each
word.*

In [ ]:
case_norm=[word.lower() for word in result]

In [ ]:
case_norm[:10]

# Tokenization and Cleaning with NLTK:

[Blog Post](https://machinelearningmastery.com/prepare-text-data-machine-learning-scikit-learn/)

In [ ]:
import nltk

##  Split into Sentences:

In [ ]:
from nltk import sent_tokenize
sents=sent_tokenize(file)

In [ ]:
sents[0]

## Split into Words:
This function splits tokens based on white space and punctuation. For example, commas and
periods are taken as separate tokens. Contractions are split apart (e.g. What’s becomes What
and ’s). Quotes are kept, and so on.

In [ ]:
from nltk import word_tokenize
tokens = word_tokenize(file)
print(tokens[:100])

## Removing Punctuation:

In [ ]:
# remove all tokens that are not alphabetic
words = [word for word in tokens if word.isalpha()]
print(words[:100])

## Removing Stop Words:

*These are the words that don't have a semantic load, they are common and can't help the alogrithm work better.*

In [ ]:
from nltk.corpus import stopwords
stop_words=stopwords.words('english')

In [ ]:
stop_words[:20]

## Creating a Pipeline for Text Cleaning:

In [ ]:
#Extracting words from a text file
words=word_tokenize(file)

#Converting to lower case
words=[word.lower() for word in words]

#Punctuation removal from words (what's becomes what and 's, this turns 's into s)
re_punc=re.compile('[{}]'.format(string.punctuation))
words=[re_punc.sub('',word) for word in words]

#Removing terms that are not alphabetic (like standalone punctuations)
words=[word for word in words if word.isalpha()]

#Removing Stop Words
words=[word for word in words if word not in stopwords.words('english')]

In [ ]:
words[:100]

*Note that still there are words like "nt" left in the list, this is not perfect and there's always something more that can be done!*

## Stem Words:

*Stemming refers to the process of reducing each word to its root or base. Some applications, like document classification, may benefit from stemming in order to both reduce the vocabulary and to focus on the sense or sentiment of a document rather than deeper meaning. a popular and long-standing method for stemming, is the Porter Stemming algorithm.*

In [ ]:
from nltk.stem.porter import PorterStemmer
words=word_tokenize(file)
ps=PorterStemmer()
stemmed_words=[ps.stem(word) for word in words]

#Note that the stemmer, turns the words to lowercase

In [ ]:
stemmed_words[:100]

*getting truly clean text is impossible, what we are really doing is the best we can based on the time, resources, and knowledge we have.*

# Encoding (word vectorizing) With SciKit-Learn:

## The Bag-of-Words Model:

*A simple and effective model for thinking about text documents in machine learning is called
the Bag-of-Words Model, or BoW.*

*Here, we are only concerned with encoding schemes that represent what words are present or the degree to which they are present in encoded documents without any information about order.*

### CountVectorizer:

*An encoded vector is returned with a length of the entire vocabulary and an integer count
for the number of times each word appeared in the document. Because these vectors will
contain a lot of zeros, we call them sparse.*

*Python provides an efficient way of handling sparsevectors in the scipy.sparse package. The vectors returned from a call to transform() will be sparse vectors, and you can transform them back to NumPy arrays to look and better understand what is going on by calling the toarray() function. Below is an example of using the CountVectorizer to tokenize, build a vocabulary, and then encode a document.*

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
# list of text documents
text = ["The quick brown fox jumped over the lazy dog."]
# create the transform
vectorizer = CountVectorizer()
# tokenize and build vocab
vectorizer.fit(text)
# summarize
print("vocabulary:{}\n".format(vectorizer.vocabulary_))
# encode document
vector = vectorizer.transform(text)
# summarize encoded vector
print("vector size:{}\n".format(vector.shape))
print(type(vector),"\n")
print("vector:\n{}".format(vector.toarray()))

*We can see that all words were made lowercase by default and that the punctuation was
ignored. These and other aspects of tokenizing can be configured.*

*Importantly, the same vectorizer can be used on documents that contain words not included
in the vocabulary. These words are ignored and no count is given in the resulting vector.*

In [ ]:
# encode another document
text2 = ["the puppy"]
vector = vectorizer.transform(text2)
print("vector:\n{}".format(vector.toarray()))

### Word Frequencies with TfidfVectorizer:

*Word counts are a good starting point, but are very basic. One issue with simple counts is that
some words like the will appear many times and their large counts will not be very meaningful
in the encoded vectors. An alternative is to calculate word frequencies, and by far the most
popular method is called TF-IDF. This is an acronym that stands for Term Frequency - Inverse
Document Frequency which are the components of the resulting scores assigned to each word*.

* **Term Frequency**: This summarizes how often a given word appears within a document.
* **Inverse Document Frequency**: This downscales words that appear a lot across documents.


**TF-IDF are word frequency scores that try to highlight
words that are more interesting, e.g. frequent in a document but not across documents.**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# list of text documents
text = ["The quick brown fox jumped over the lazy dog.",
"The dog.",
"The fox!"]
# create the transform
vectorizer = TfidfVectorizer()
# tokenize and build vocab
vectorizer.fit(text)
# summarize
print(vectorizer.vocabulary_)
print(vectorizer.idf_)
# encode document
vector = vectorizer.transform([text[0]])
# summarize encoded vector
print(vector.shape)
print(vector.toarray())


**Note**: *The scores are normalized to values between 0 and 1.*

**Note**: *Alternately, if you already have a
learned CountVectorizer, you can use it with a TfidfTransformer to just calculate the inverse
document frequencies and start encoding documents.*

###  Hashing with HashingVectorizer:

*Counts and frequencies can be very useful, but one limitation of these methods is that the
vocabulary can become very large.This, in turn, will require large vectors for encoding
documents and impose large requirements on memory and slow down algorithms. A clever work
around is to use a one way hash of words to convert them to integers. The clever part is that
no vocabulary is required and you can choose an arbitrary-long fixed length vector. A downside
is that the hash is a one-way function so there is no way to convert the encoding back to a word (which may not matter for many supervised learning tasks).*

**Note that this vectorizer does not require a call to fit on the training data documents.**

**Note:** *we should be careful about hash collision. it happens when two words have the same vector. Take the well-known hash function CRC32, for example. If you feed this function the two strings “plumless” and “buckeroo”, it generates the same value. This is known as a hash collision.* 

***Below we can see hash collision happening, because Every hash function with more inputs than outputs will necessarily have collisions. (number of words in the list > n_features in hashingvectorizer)***

**HASH COLLISION:**

In [ ]:
from sklearn.feature_extraction.text import HashingVectorizer
# list of text documents
text = ["The", "quick", "brown", "fox", "jumped", "over" ,"the" ,"lazy" ,"dog"]
# create the transform
vectorizer = HashingVectorizer(n_features=3)
# encode document
vector = vectorizer.transform(text)
# summarize encoded vector
print(vector.shape)
print(vector.toarray())

**NO HASH COLLISION:**

In [ ]:
from sklearn.feature_extraction.text import HashingVectorizer
# list of text documents
text = ["The", "quick", "brown", "fox", "jumped", "over" ,"the" ,"lazy" ,"dog"]
# create the transform
vectorizer = HashingVectorizer(n_features=10)
# encode document
vector = vectorizer.transform(text)
# summarize encoded vector
print(vector.shape)
print(vector.toarray())

#  Prepare Text Data With Keras
*The Keras deep learning library provides some basic tools to help you
prepare your text data.*

[Blog Post](https://machinelearningmastery.com/prepare-text-data-deep-learning-keras/)


## Text to word sequence:
*this function mainly does 3 things:*

*1. Splitting the text by space*

*2. Removing punctuations*

*3. Lowercasing the words*

In [ ]:
from keras.preprocessing.text import text_to_word_sequence
# define the document
text = 'The quick brown fox, jumped over the lazy dog.'
# tokenize the document
result = text_to_word_sequence(text)
print(result)


# Encoding (Word Vectorizing) With Keras:

##  Encoding with one hot:

*Keras provides the one_hot() function that you
can use to tokenize and integer encode a text document in one step. The name suggests that it
will create a one hot encoding of the document, **which is not the case**. Instead, the function
is a wrapper for the hashing_trick() function*

**NOTE: The use of a hash function means that
there may be collisions and not all words will be assigned unique integer values.**

*As with the
text_to_word_sequence() function in the previous section, the one_hot() function will make
the text lower case, filter out punctuation, and split words based on white space.
In addition to the text, the vocabulary size (total words) must be specified. This could be the
total number of words in the document or more if you intend to encode additional documents
that contains additional words. The size of the vocabulary defines the hashing space from which
words are hashed.*

In [ ]:
from keras.preprocessing.text import one_hot
from keras.preprocessing.text import text_to_word_sequence
# define the document
text = 'The quick brown fox jumped over the lazy dog.'
# estimate the size of the vocabulary
words = set(text_to_word_sequence(text))
vocab_size = len(words)
print(vocab_size)
# integer encode the document
result = one_hot(text, round(vocab_size*10))
print(result)

## Hash Encoding with Hashing Trick:

*Keras provides the hashing_trick() function that tokenizes and then integer encodes the
document, just like the one_hot() function. It provides more flexibility, allowing you to specify
the hash function as either hash (the default) or other hash functions such as the built in md5
function or your own function.*

In [ ]:
from keras.preprocessing.text import hashing_trick
from keras.preprocessing.text import text_to_word_sequence
# define the document
text = 'The quick brown fox jumped over the lazy dog.'
# estimate the size of the vocabulary
words = set(text_to_word_sequence(text))
vocab_size = len(words)
print(vocab_size)
# integer encode the document
result = hashing_trick(text, round(vocab_size*10), hash_function='md5')
print(result)


## Tokenizer API:

*Keras provides
the Tokenizer class for preparing text documents for deep learning. The Tokenizer must be
constructed and then fit on either raw text documents or integer encoded text documents. This may be the preferred approach for large projects*

In [ ]:
from keras.preprocessing.text import Tokenizer
# define 5 documents
docs = ['Well done!',
'Good work',
'Great effort',
'nice work',
'Excellent']
# create the tokenizer
t = Tokenizer()
# fit the tokenizer on the documents
t.fit_on_texts(docs)

*Once fit, the Tokenizer provides 4 attributes that you can use to query what has been
learned about your documents:*
*   word counts
*   word docs
*   word index
*   document count

In [ ]:
# summarize what was learned
print("Ordered Word count in docs: \n",t.word_counts,'\n')
print("Word count in docs: \n",t.word_docs)
print("Number of documents in docs: \n",t.document_count,'\n')
print("Unique Integer assigned to each word: \n",t.word_index,'\n')

*The texts_to_matrix() function on the Tokenizer can be used to
create one vector per document provided per input. The length of the vectors is the total size
of the vocabulary. This function provides a suite of standard bag-of-words model text encoding
schemes that can be provided via a mode argument to the function.*

In [ ]:
# integer encode documents
encoded_docs = t.texts_to_matrix(docs, mode='binary')  
print(encoded_docs)
#there are 9 columns, I assume it's because the integers
#assigned to the words, start from 1 and in array indexing, we start from 0

"mode" can be:
*  **binary**: Whether or not each word is present in the document. This is the default.
*  **count**: The count of each word in the document.
*  **tfidf**: The Text Frequency-Inverse DocumentFrequency (TF-IDF) scoring for each word in the document.
*  **freq**: The frequency of each word as a ratio of words within each document.


# Thank You For Reading!